# HindiMix-Noisy — All CPU Baselines (Colab)
Runs **all CPU baselines** on all 4 noise levels — authoritative source of results.

| Model | Noise levels |
|-------|--------------|
| TF-IDF + SVM | clean, low, medium, high |
| TF-IDF + LR  | clean, low, medium, high |
| CharCNN      | clean, low, medium, high |

**Runtime: CPU is fine.** (~25-30 min total)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_PROJECT = '/content/drive/MyDrive/hindiMix-noisy'
RESULTS_DIR   = f'{DRIVE_PROJECT}/results/tables'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Results will save to:', RESULTS_DIR)

## Step 2: Upload data/final/ CSVs from your PC
Click file picker → select **all 6 CSVs** from your local `data/final/`:
`train.csv, val.csv, test_clean.csv, test_noisy_low.csv, test_noisy_medium.csv, test_noisy_high.csv`

In [ ]:
from google.colab import files
import os

DATA_DIR = '/content/data/final'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'{DATA_DIR}/{fname}', 'wb') as f:
        f.write(content)

print('\nUploaded files:')
for f in sorted(os.listdir(DATA_DIR)):
    rows = sum(1 for _ in open(f'{DATA_DIR}/{f}')) - 1
    print(f'  {f}: {rows:,} rows')

## Step 3: Setup

In [ ]:
import os, json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

NOISE_LEVELS = ['clean', 'low', 'medium', 'high']

def load_data(noise_level):
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text','label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text','label'])
    if noise_level != 'all':
        train_df = train_df[train_df['noise_level'].isin(['clean', noise_level])]
    test_path = f'{DATA_DIR}/test_clean.csv' if noise_level == 'clean' \
                else f'{DATA_DIR}/test_noisy_{noise_level}.csv'
    if not os.path.exists(test_path):
        test_path = f'{DATA_DIR}/test_clean.csv'
    test_df = pd.read_csv(test_path).dropna(subset=['text','label'])
    return train_df, val_df, test_df

def save_result(result, name):
    out = f'{RESULTS_DIR}/{name}.json'
    with open(out, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'    Saved → {out}')

all_results = []
print('Setup complete.')

---
## Step 4: TF-IDF + SVM and TF-IDF + LR (all noise levels)
~2 minutes total

In [ ]:
def build_tfidf_pipeline(clf_type):
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                          max_features=100_000, sublinear_tf=True)
    clf = LinearSVC(C=1.0, max_iter=2000) if clf_type == 'svm' \
          else LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', n_jobs=-1)
    return Pipeline([('tfidf', vec), ('clf', clf)])

print('── TF-IDF Baselines ──')
for noise in NOISE_LEVELS:
    train_df, val_df, test_df = load_data(noise)
    X_tr, y_tr = train_df['text'].astype(str).tolist(), train_df['label'].tolist()
    X_va, y_va = val_df['text'].astype(str).tolist(),   val_df['label'].tolist()
    X_te, y_te = test_df['text'].astype(str).tolist(),  test_df['label'].tolist()

    for clf_type in ['svm', 'lr']:
        pipe = build_tfidf_pipeline(clf_type)
        pipe.fit(X_tr, y_tr)
        val_f1  = f1_score(y_va, pipe.predict(X_va), average='macro')
        test_f1 = f1_score(y_te, pipe.predict(X_te), average='macro')
        result = {
            'model': f'tfidf_{clf_type}', 'noise_level': noise,
            'val_f1_macro': round(val_f1, 4), 'test_f1_macro': round(test_f1, 4)
        }
        save_result(result, f'tfidf_{clf_type}_{noise}')
        all_results.append(result)
        print(f'  tfidf_{clf_type} | {noise:8s} | val: {val_f1:.4f} | test: {test_f1:.4f}')

print('\nTF-IDF done!')

---
## Step 5: CharCNN (all noise levels)
~5-7 minutes per noise level (~25 min total)

In [ ]:
CHARS = list(' abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?@#&\'- ') + ['<PAD>','<UNK>']
CHAR2IDX  = {c: i for i, c in enumerate(CHARS)}
PAD_IDX   = CHAR2IDX['<PAD>']
UNK_IDX   = CHAR2IDX['<UNK>']
VOCAB_SIZE = len(CHARS)

def text_to_ids(text, max_len=300):
    ids = [CHAR2IDX.get(c, UNK_IDX) for c in str(text)[:max_len]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

class CharDataset(Dataset):
    def __init__(self, texts, labels):
        self.data   = [torch.tensor(text_to_ids(t), dtype=torch.long) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        return self.data[i], torch.tensor(self.labels[i], dtype=torch.long)

class CharCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, 64, padding_idx=PAD_IDX)
        self.convs  = nn.ModuleList([nn.Conv1d(64, 128, fs) for fs in (3, 4, 5)])
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(128 * 3, 2)
    def forward(self, x):
        e = self.embedding(x).permute(0, 2, 1)
        pooled = [F.max_pool1d(F.relu(c(e)), c(e).size(2)).squeeze(2) for c in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

print('── CharCNN Baselines ──')
for noise in NOISE_LEVELS:
    print(f'\n  noise={noise}')
    train_df, val_df, test_df = load_data(noise)

    train_loader = DataLoader(CharDataset(train_df['text'].tolist(), train_df['label'].tolist()),
                              batch_size=128, shuffle=True)
    val_loader   = DataLoader(CharDataset(val_df['text'].tolist(),   val_df['label'].tolist()),
                              batch_size=128)
    test_loader  = DataLoader(CharDataset(test_df['text'].tolist(),  test_df['label'].tolist()),
                              batch_size=128)

    model     = CharCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state = 0, None

    for epoch in range(10):
        model.train()
        for X, y in tqdm(train_loader, desc=f'    Epoch {epoch+1}/10', leave=False):
            optimizer.zero_grad()
            criterion(model(X.to(DEVICE)), y.to(DEVICE)).backward()
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for X, y in val_loader:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().tolist())
                targets.extend(y.tolist())
        val_f1 = f1_score(targets, preds, average='macro')
        print(f'    Epoch {epoch+1:2d} | Val F1: {val_f1:.4f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, y in test_loader:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().tolist())
            targets.extend(y.tolist())
    test_f1 = f1_score(targets, preds, average='macro')
    print(f'  Test F1: {test_f1:.4f}')
    print(classification_report(targets, preds, target_names=['Non-hate','Hate']))

    result = {
        'model': 'CharCNN', 'noise_level': noise,
        'best_val_f1': round(best_val_f1, 4), 'test_f1_macro': round(test_f1, 4),
        'epochs': 10, 'lr': 0.001
    }
    save_result(result, f'char_cnn_{noise}')
    all_results.append(result)

print('\nCharCNN all done!')

## Step 6: Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
pivot = df.pivot_table(index='model', columns='noise_level', values='test_f1_macro')
for col in ['clean','low','medium','high']:
    if col not in pivot.columns:
        pivot[col] = float('nan')
pivot = pivot[['clean','low','medium','high']]
pivot['degradation'] = pivot['clean'] - pivot['high']
print(pivot.round(4).to_string())

pivot.to_csv(f'{RESULTS_DIR}/CPU_RESULTS.csv')
print('\nSaved CPU_RESULTS.csv to Drive')

## Step 7: Download zip to your PC

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/cpu_results', 'zip', RESULTS_DIR)
files.download('/content/cpu_results.zip')
print('Downloaded cpu_results.zip')
print('On your PC run:')
print('  python scripts/sync_from_drive.py --zip /path/to/cpu_results.zip')